In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS


In [ ]:
document = PyMuPDFLoader('attention-is-all-you-need-Paper.pdf')
pages = document.load()



In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap =  60
)

chunks =  text_splitter.split_documents(pages)

In [10]:
embedding = HuggingFaceEmbeddings(
    model = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
vector_db = FAISS.from_documents(
    chunks,
    embedding
)

<h1>RETRIVER</h1>

In [26]:
retriver = vector_db.as_retriever(
    search_kwargd =  {'k':5}
)




<h1>PROMPT</h1>

In [27]:
prompt =  ChatPromptTemplate.from_template(
    '''
    Answer the questions from the given context.
    if the question is out of the context. just say: 'Question out of context'.

    context
    {context}

    question
    {question}
    
'''
)

<h1>MOdel</H1>

In [28]:
model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    temperature = 0
)

<h1>QUERY</h1>

In [29]:
question = input('Enter your question: ')

<h1>Retriver invoke</h1>

In [30]:
retriver_invoke = retriver.invoke(question)

<h1>Context</h1>

In [31]:
context= '\n\n'.join(
    pages.page_content
    for pages in retriver_invoke
)

<h1>Prompt invoke</h1>

In [32]:
prompt_invoke = prompt.invoke(
    {
        'context':context,
        'question':question
    }
)

<h1>MODEL INVOKE</h1>

In [33]:
response = model.invoke(prompt_invoke)

e:\GEN-AI-PROJECTS\genai-env\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [34]:
print(response.text())

Based on the provided context, the **decoder** is composed of a stack of $N = 6$ identical layers. 

Key features of the decoder described in the text include:
* **Three Sub-layers:** In addition to the two sub-layers found in each encoder layer, the decoder inserts a third sub-layer that performs multi-head attention over the output of the encoder stack.
* **Residual Connections and Normalization:** It employs residual connections around each of the sub-layers, followed by layer normalization.
* **Masked Self-Attention:** The self-attention sub-layer is modified with masking to prevent positions from attending to subsequent positions, ensuring predictions for position $i$ depend only on known outputs at positions less than $i$.
